In [3]:
import face_recognition
import os
import numpy as np
import pickle
import cv2

In [4]:
import dlib
print("DLIB CUDA Enabled:", dlib.cuda.get_num_devices() > 0)

DLIB CUDA Enabled: True


# Step 1: Encode Your Face Dataset

In [4]:
DATASET_PATH = "./Students Faces/"  # Path to your dataset
ENCODINGS_FILE = "face_encodings2.pkl"  # File to save known encodings

known_face_encodings = []  # List to store face encodings
known_face_names = []  # List to store corresponding names

# Loop through each person in the dataset
for person_name in os.listdir(DATASET_PATH):
    person_folder = os.path.join(DATASET_PATH, person_name)
    
    if not os.path.isdir(person_folder):
        continue  # Skip non-folder files
    
    # Loop through each image of the person
    for filename in os.listdir(person_folder):
        img_path = os.path.join(person_folder, filename)
        
        # Load the image
        image = face_recognition.load_image_file(img_path)
        
        # Encode the face (assuming one face per image)
        encodings = face_recognition.face_encodings(image)
        
        if len(encodings) > 0:
            known_face_encodings.append(encodings[0])  # Add first face encoding
            known_face_names.append(person_name)  # Store person name
        else:
            print(f"Warning: No face found in {img_path}")

# Save the encodings to a file for future use
with open(ENCODINGS_FILE, "wb") as f:
    pickle.dump((known_face_encodings, known_face_names), f)

print("✅ Dataset encoding complete. Encodings saved to", ENCODINGS_FILE)

✅ Dataset encoding complete. Encodings saved to face_encodings2.pkl


# Step 2: Recognize Faces from a New Image

In [3]:
ENCODINGS_FILE = "face_encodings2.pkl"

# Load stored encodings
with open(ENCODINGS_FILE, "rb") as f:
    known_face_encodings, known_face_names = pickle.load(f)

def recognize_face(image_path, output_path="output_detected.jpg", threshold=0.5):
    # Load the test image
    test_image = face_recognition.load_image_file(image_path)

    # Get face locations and encodings
    face_locations = face_recognition.face_locations(test_image)
    face_encodings = face_recognition.face_encodings(test_image, face_locations)

    # Convert image to OpenCV format for visualization
    test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

    recognized_faces = []

    for face_encoding, face_location in zip(face_encodings, face_locations):
        name = "Unknown"
        best_distance = 1.0  # Initialize with a high distance value

        # Compare face encoding with known encodings
        distances = face_recognition.face_distance(known_face_encodings, face_encoding)

        if len(distances) > 0:
            # Find the best match
            best_match_index = np.argmin(distances)
            best_distance = distances[best_match_index]

            if best_distance < threshold:
                name = known_face_names[best_match_index]

        recognized_faces.append((name, best_distance))

        # Draw bounding box around face
        top, right, bottom, left = face_location
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(test_image_cv, (left, top), (right, bottom), color, 2)
        cv2.putText(test_image_cv, f"{name} ({best_distance:.2f})", (left, top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # **Save the output image with detection results**
    cv2.imwrite(output_path, test_image_cv)
    print(f"Image saved as {output_path}")

    # Show the image (Optional)
    cv2.imshow("Recognized Faces", test_image_cv)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return recognized_faces

# Example usage
image_path = "Test Images/DM Class.jpg"  
output_path = "Detected_Faces.jpg"  # Specify output file name
results = recognize_face(image_path, output_path)

for name, confidence in results:
    print(f"Detected: {name} (Distance: {confidence:.2f})")

Image saved as Detected_Faces.jpg
Detected: Unknown (Distance: 0.67)
Detected: Fatemeh Ghoreh Zadeh (Distance: 0.47)
Detected: Elahe Heydari (Distance: 0.43)
Detected: Mobina Kazemi (Distance: 0.40)
Detected: Unknown (Distance: 0.61)
Detected: Unknown (Distance: 0.59)
Detected: Fahimeh Motian (Distance: 0.47)
Detected: Mohammad Reza Hadizadeh (Distance: 0.25)
Detected: Unknown (Distance: 0.55)
Detected: Unknown (Distance: 0.64)
Detected: Unknown (Distance: 0.62)
Detected: Unknown (Distance: 0.68)
Detected: Hossein Foroughi (Distance: 0.44)
Detected: S. Mohammad Hosseini Renani (Distance: 0.45)
Detected: Mohammad Arab Jahvani (Distance: 0.46)
Detected: Unknown (Distance: 0.53)
Detected: Mostafa Karami (Distance: 0.43)
Detected: Unknown (Distance: 0.64)
Detected: Arman Shoja (Distance: 0.45)
Detected: Marzi Matlabpour (Distance: 0.45)
Detected: Reza Gholampour (Distance: 0.46)
Detected: Unknown (Distance: 0.62)
Detected: Unknown (Distance: 0.53)
Detected: Unknown (Distance: 0.57)


# Real-Time Face Recognition Code

In [1]:
import cv2
import numpy as np
import pickle
import face_recognition
import threading

ENCODINGS_FILE = "face_encodings.pkl"

# Load stored face encodings
with open(ENCODINGS_FILE, "rb") as f:
    known_face_encodings, known_face_names = pickle.load(f)

# IP Webcam URL (replace with your phone's actual IP)
IP_CAMERA_URL = "http://192.168.43.1:8080/video"  # Update with your actual IP
video_capture = cv2.VideoCapture(IP_CAMERA_URL)
video_capture.set(cv2.CAP_PROP_FPS, 30)  # Try setting FPS to 30

# Multi-threading for faster frame capture
frame_lock = threading.Lock()
latest_frame = None

def capture_frame():
    global latest_frame
    while True:
        ret, frame = video_capture.read()
        if ret:
            with frame_lock:
                latest_frame = frame

threading.Thread(target=capture_frame, daemon=True).start()

print("📷 Connected to IP Webcam. Press 'q' to exit.")
frame_count = 0

while True:
    with frame_lock:
        if latest_frame is None:
            continue
        frame = latest_frame.copy()

    # Resize frame for faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Detect face locations using HOG model (faster)
    face_locations = face_recognition.face_locations(rgb_frame, model="hog")

    frame_count += 1
    if frame_count % 3 == 0:  # Only encode every 3rd frame
        face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)
    else:
        face_encodings = []

    for face_encoding, face_location in zip(face_encodings, face_locations):
        name = "Unknown"
        best_distance = 1.0  # Initialize with a high value

        # Compare face encoding with known encodings
        distances = face_recognition.face_distance(known_face_encodings, face_encoding)

        if len(distances) > 0:
            best_match_index = np.argmin(distances)
            best_distance = distances[best_match_index]

            if best_distance < 0.5:
                name = known_face_names[best_match_index]

        # Scale face location back up since we resized the frame
        top, right, bottom, left = [int(coord * 2) for coord in face_location]

        # Draw bounding box and name
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        cv2.putText(frame, f"{name} ({best_distance:.2f})", (left, top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Show the video frame
    display_frame = cv2.resize(frame, (800, 800))  # Adjust the window size (width, height)
    cv2.imshow("Real-Time Face Recognition (IP Webcam)", display_frame)
    
    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release video capture and close windows
video_capture.release()
cv2.destroyAllWindows()


📷 Connected to IP Webcam. Press 'q' to exit.


[mjpeg @ 0x55e8e68fa940] overread 8
[mjpeg @ 0x55e8e68fa940] overread 8
